In [10]:
numCores = 8
print(f"Number of cores: {numCores}")

Number of cores: 8


In [11]:
import os
from glob import glob

# Topology file directory
topo_dir = "Data/Dorothea"#"pGRiNS/TOPOS"#
# Directory to store simulation results
sim_save_dir = "SimulResults"
# Create output directory if it does not exist
os.makedirs(sim_save_dir, exist_ok=True)

In [14]:
topo_files = sorted(glob(f"{topo_dir}/*.topo"))[6:7]
print(f"Number of topology files: {len(topo_files)}")
print(topo_files)

Number of topology files: 1
['Data/Dorothea/dorothea_trunc.topo']


In [15]:
import sys
sys.path.append("./pGRiNS")

# RACIPE:

In [16]:
num_replicates = 1
num_params = 10000
num_init_conds = 100
sampling_method = "Uniform"
print(f"Number of replicates: {num_replicates}")
print(f"Number of parameters: {num_params}")
print(f"Number of initial conditions: {num_init_conds}\n")

Number of replicates: 1
Number of parameters: 10000
Number of initial conditions: 100



In [17]:
import jax.numpy as jnp
import grins.racipe_run as racipe
import multiprocessing as mp
mp.set_start_method("spawn", force=True)
# Start multiprocessing pool
pool = mp.Pool(numCores)
print("Generating Parameter and Initial Condition files...")

# Parallel execution of file generation
pool.starmap(
    racipe.gen_topo_param_files,
    [
        (
            topo_file,
            sim_save_dir,
            num_replicates,
            num_params,
            num_init_conds,
            sampling_method
        )
        for topo_file in topo_files
    ],
)
print("Parameter and Initial Condition files generated.\n")

# Close multiprocessing pool
pool.close()

Generating Parameter and Initial Condition files...
Parameter and Intial Condition files generated for dorothea_trunc
Parameter and Initial Condition files generated.



In [ ]:

from importlib import reload
reload(racipe)
for topo_file in topo_files:
    # Generate parameters using Sobol sampling (optional - if the paramters are not already generated in parallel)
    """
    racipe.gen_topo_param_files(
        topo_file,
        sim_save_dir,
        num_replicates,
        num_params,
        num_init_conds,
        sampling_method="Uniform",
    )
    """
    
    
    racipe.run_all_replicates(
        topo_file,
        sim_save_dir,
        #tsteps=jnp.array([25.0, 75.0, 100.0]), # Run time-series simulations
        max_steps=1024, # originally 2048
        batch_size=6000,
    )


Loading ODE system from: SimulResults/dorothea_trunc
Number of combinations to simulate: 1000000
Running steady state simulations for replicate: 001
Time taken for replicate 001: 455.31147718429565
Normalising and Discretising the solutions


/home/mi/sommereg03/bsc_thesis_pGRiNS/pGRiNS/grins/racipe_run.py:814: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  deg_cols = [f"Deg_{node}" for node in node_cols]
/home/mi/sommereg03/bsc_thesis_pGRiNS/pGRiNS/grins/racipe_run.py:814: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  deg_cols = [f"Deg_{node}" for node in node_cols]
/home/mi/sommereg03/bsc_thesis_pGRiNS/pGRiNS/grins/racipe_run.py:814: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor p

Simulation completed for replicate: 001



# Boolise:

In [6]:
import jax.numpy as jnp
import grins.ising_bool as ising_bool
replacement_values = jnp.array([-1, 1])
max_steps = 100
print(f"Number of steps: {max_steps}")
num_initial_conditions = 2**14
print(f"Number of initial conditions: {num_initial_conditions}")
batch_size = 2**10
num_replicates = 3
save_dir = "SimulResultsB"

Number of steps: 100
Number of initial conditions: 16384


In [9]:
from importlib import reload
reload(ising_bool)
for topo_file in topo_files:
    """
    ising_bool.run_all_replicates_ising(
        topo_file,
        num_initial_conditions=num_initial_conditions,
        batch_size=batch_size,
        save_dir=save_dir,
        mode="sync",
        packbits=True,
        num_replicates=num_replicates
    )
    """
    ising_bool.run_all_replicates_ising(
        topo_file,
        num_initial_conditions=num_initial_conditions,
        batch_size=batch_size,
        save_dir=save_dir,
        mode="async",
        packbits=True,
    )

Topology: Data/Dorothea/dorothea_tf2_abc.topo
Running async simulations for the network: Data/Dorothea/dorothea_tf2_abc.topo
Simulation time for async mode: 2.41 seconds
Running async simulations for the network: Data/Dorothea/dorothea_tf2_abc.topo
Simulation time for async mode: 1.69 seconds
Running async simulations for the network: Data/Dorothea/dorothea_tf2_abc.topo
Simulation time for async mode: 1.69 seconds
